## RA-CNN Training 코드

In [5]:
BATCH_SIZE = 16

### data: CUB-200-2011

#### 데이터셋 로드 from datasets (허깅페이스)

In [2]:
from datasets import load_dataset


dataset = load_dataset(path="bentrevett/caltech-ucsd-birds-200-2011")

/home/jongha/projects/eye-openning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 데이터셋 세이브 to 로컬

In [3]:
from pathlib import Path


dataset_save_path = Path("./data/CUB-200-2011/")
dataset.save_to_disk(dataset_save_path)

print(f'경로: "./{dataset_save_path}"')

Saving the dataset (2/2 shards): 100%|██████████| 5794/5794 [00:02<00:00, 2725.70 examples/s]

경로: "./data/CUB-200-2011"


#### 데이터셋 로드 from 로컬

In [4]:
from pathlib import Path
from datasets import load_from_disk

cub_dataset_save_path = Path("./data/CUB-200-2011/")
cub_dataset = load_from_disk(dataset_path=cub_dataset_save_path)

print(cub_dataset["train"][2])

{'image': <PIL.Image.Image image mode=RGB size=500x416 at 0x7857D3FAD670>, 'label': 0, 'bbox': [112.0, 76.0, 333.0, 265.0]}


#### 로더 정의

In [5]:
from torchvision import transforms
from torch.utils.data import DataLoader


cub_train_set = cub_dataset["train"]
cub_test_set = cub_dataset["test"]


train_transform = transforms.Compose([
    transforms.Resize(size=(512, 512)),
    transforms.RandomCrop(size=448),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize(size=(512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])


def apply_train_transform(example):
    example["image"] = [train_transform(image.convert("RGB")) for image in example["image"]]
    
    return example

def apply_test_transform(example):
    example["image"] = [test_transform(image.convert("RGB")) for image in example["image"]]
    
    return example


def get_cub_loader(batch_size):
    train_set = cub_train_set.with_transform(apply_train_transform)
    test_set = cub_test_set.with_transform(apply_test_transform)
    
    train_loader = DataLoader(
        dataset=train_set,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        dataset=test_set,
        batch_size=batch_size,
    )

    return train_loader, test_loader

In [6]:
train_loader, test_loader = get_cub_loader(batch_size=BATCH_SIZE)

### data: FGVC-Aircraft

#### 데이터 다운로드 from kagglehub (캐글)

In [ ]:
import kagglehub


path = kagglehub.dataset_download("seryouxblaster764/fgvc-aircraft")

print("Path to dataset files:", path)

Path to dataset files: /home/jongha/.cache/kagglehub/datasets/seryouxblaster764/fgvc-aircraft/versions/2


#### 데이터 로더 정의

In [8]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path


aircraft_csv_dir = Path("./data/FGVC-Aircraft")
aircraft_images_dir = Path("./data/FGVC-Aircraft/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data/images")


class AircraftDataset(Dataset):
    
    def __init__(self, csv_file: Path, images_dir: Path, transform):
        super().__init__()
        self.dataframe = pd.read_csv(
            filepath_or_buffer=aircraft_csv_dir / csv_file, 
            encoding="utf-8",
        )
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        
        return len(self.dataframe)

    def __getitem__(self, index):
        image_name = self.dataframe.iloc[index]["filename"]
        image_path = self.images_dir / image_name
        
        image = Image.open(fp=image_path)
        label = int(self.dataframe.iloc[index]["Labels"])

        if self.transform:
            image = self.transform(image)

        return {
            "image": image, 
            "label": label,
        }

def get_aircraft_dataloaders(batch_size):    
    train_transform = transforms.Compose([
        transforms.Resize(size=(512, 512)),
        transforms.RandomCrop(size=448),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    test_transform = transforms.Compose([
        transforms.Resize(size=(512, 512)),
        transforms.CenterCrop(size=448),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])    

    train_dataset = AircraftDataset(
        csv_file="train.csv",
        images_dir=aircraft_images_dir,
        transform=train_transform,
    )
    test_dataset = AircraftDataset(
        csv_file="test.csv",
        images_dir=aircraft_images_dir,
        transform=test_transform,
    )

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=batch_size,
    )

    return train_loader, test_loader

In [9]:
train_loader, test_loader = get_aircraft_dataloaders(batch_size=BATCH_SIZE)

### data: Stanford-Cars

#### 데이터 다운로드 from kagglehub (캐글)

In [ ]:
import kagglehub


path = kagglehub.dataset_download("eduardo4jesus/stanford-cars-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.82G/1.82G [02:15<00:00, 14.4MB/s]

Extracting files...


Path to dataset files: /home/jongha/.cache/kagglehub/datasets/eduardo4jesus/stanford-cars-dataset/versions/1


In [3]:
import scipy.io


cars_meta = scipy.io.loadmat(file_name="data/Stanford-Cars/car_devkit/devkit/cars_meta.mat")
cars_train_annos = scipy.io.loadmat(file_name="data/Stanford-Cars/car_devkit/devkit/cars_train_annos.mat")
cars_test_annos_withlabels = scipy.io.loadmat(file_name="data/Stanford-Cars/car_devkit/devkit/cars_test_annos_withlabels.mat")

#### 데이터 로더 정의

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image


cars_train_images_dir = Path("data/Stanford-Cars/cars_train/cars_train/")
cars_test_images_dir = Path("data/Stanford-Cars/cars_test/cars_test/")

    
class CarsDataset(Dataset):
    
    def __init__(self, image_dir: Path, train_or_test: str, transform: transforms = None):
        super().__init__()
        self.transform=transform
        self.image_paths = sorted(image_dir.glob(pattern="*.jpg"))
        self.train_or_test = train_or_test
        self.train_labels = [int(cars_train_annos["annotations"][0][i][4][0][0]) - 1 \
            for i in range(len(cars_train_annos["annotations"][0]))]
        self.test_labels = [int(cars_test_annos_withlabels["annotations"][0][i][4][0][0]) - 1 \
            for i in range(len(cars_test_annos_withlabels["annotations"][0]))]
        
    def __len__(self) -> int:

        return len(self.image_paths)

    def __getitem__(self, index: int) -> dict:
        image = Image.open(fp=self.image_paths[index])

        if self.train_or_test == "train":
            label = self.train_labels[index]
        else:
            label = self.test_labels[index]

        if self.transform:
            image = self.transform(image.convert(mode="RGB"))
            
        return {
            "image": image, 
            "label": label,
        }
        

def cars_loader(batch_size: int) -> DataLoader:
    train_set = CarsDataset(
        image_dir=cars_train_images_dir,
        train_or_test="train",
        transform=train_transform,
    )
    test_set = CarsDataset(
        image_dir=cars_test_images_dir,
        train_or_test="test",
        transform=test_transform,
    )

    train_loader = DataLoader(
        dataset=train_set,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        dataset=test_set,
        batch_size=batch_size,
    ) 
    
    return train_loader, test_loader

In [20]:
train_loader, test_loader = cars_loader(batch_size=BATCH_SIZE)

### 모델 정의

#### Attention Proposal Network

In [3]:
import torch
from torch import nn
import torch.nn.functional as F


class APN(nn.Module):
    
    def __init__(self, in_features):
        super().__init__()

        self.fc1 = nn.Linear(in_features, 1024)
        self.fc2 = nn.Linear(1024, 3)

        nn.init.zeros_(tensor=self.fc2.weight)
        self.fc2.bias.data.copy_(torch.tensor([0.0, 0.0, 0.5]))

    def foward(self, x):
        x = x.view(x.size(dim=0), -1)

        x = self.fc1(x)
        x = F.relu(input=x)        
        x = self.fc2(x)

        tx = x[:, 0]
        ty = x[:, 1]
        tl = x[:, 2]

        tx = torch.tanh(tx)
        ty = torch.tanh(ty)
        tl = torch.sigmoid(tl)

        return tx, ty, tl

#### Crop Function

In [ ]:
def crop_image(image, tx, ty, tl):
    b, _, _, _ = image.size()

    theta = torch.zeros(size=(b, 2, 3), dtype=image.dtype, device=image.device)
    theta[:, 0, 0] = tl
    theta[:, 0, 2] = tx
    theta[:, 1, 1] = tl
    theta[:, 1, 2] = ty

    grid = F.affine_grid(theta=theta, size=image.size(), align_corners=False)
    cropped_image = F.grid_sample(input=image, grid=grid, align_corners=False)

    return cropped_image

#### RA-CNN

In [ ]:
from torchvision.models import vgg19_bn, VGG19_BN_Weights


class RACNN(nn.Module):

    def __init__(self, num_classes: int = 200):
        super().__init__()

        base = vgg19_bn(weights=VGG19_BN_Weights.IMAGENET1K_V1)
        self.features_1 = base.features
        self.classifier_1 = nn.Sequential([
            nn.AdaptiveAvgPool2d((7, 7)), 
            nn.Flatten(),              
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),                  
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        ])
        self.apn_1 = APN(in_features=512 * 14 * 14)

        base_2 = vgg19_bn(weights=VGG19_BN_Weights.IMAGENET1K_V1)
        self.features_2 = base_2.features
        self.classifier_2 = nn.Sequential(*list(self.classifier_1.children())[:-1], [nn.Linear(4096, num_classes)])
        self.apn_2 = APN(in_features=512 * 14 * 14)

        base_3 = vgg19_bn(weight=VGG19_BN_Weights.IMAGENET1K_V1)
        self.features_3 = base_3.features
        self.classifier_3 = nn.Sequential(*list(self.classifier_1.children())[:-1], [nn.Linear(4096, num_classes)])

    def forward(self, x):
        feat_1 = self.features_1(x)
        logits_1 = self.classifier_1(feat_1)
        tx1, ty1, tl1 = self.apn_1(feat_1)
        crop_1 = crop_image(x, tx=tx1, ty=ty1, tl=tl1)
        resized_crop_1 = F.interpolate(input=crop_1, size=(448, 448), mode="bilinear", align_corners=False)

        feat_2 = self.features_2(resized_crop_1)
        logits_2 = self.classifier_2(feat_2)
        tx2, ty2, tl2 = self.apn_2(feat_2)
        crop_2 = crop_image(image=crop_1, tx=tx2, ty=ty2, tl=tl2)
        resized_crop_2 = F.interpolate(input=crop_2, size=(448, 448), mode="bilinear", align_corners=False)

        feat_3 = self.features_3(resized_crop_2)
        logits_3 = self.classifier_3(feat_3)

        return [logits_1, logits_2, logits_3], [crop_1, crop_2]

### 학습

#### Inter loss

In [9]:
import torch
import torch.nn.functional as F
from torch import nn, optim


def rank_loss(logits_coarse, logits_fine, labels, margin=0.05) -> torch.Tensor:
    probabillity_coarse = F.softmax(input=logits_coarse, dim=1)
    probabillity_fine = F.softmax(input=logits_fine, dim=1)

    batch_size = labels.size(dim=0)
    batch_indices = torch.arange(end=batch_size, device=labels.device)

    probabillity_coarse_correct = probabillity_coarse[batch_indices, labels]
    probabillity_fine_correct = probabillity_fine[batch_indices, labels]

    target = torch.ones(size=batch_size, device=labels.device)

    loss = F.margin_ranking_loss(
        input1=probabillity_fine_correct, 
        input2=probabillity_coarse_correct, 
        target=target,
        margin=margin,
    )

    return loss

#### Train (Intra loss + Inter loss)

In [11]:

def train_racnn(model, train_loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    correct1, correct2, correct3 = 0, 0, 0
    total = 0
    for index, batch in enumerate(train_loader):
        images = batch["images"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits, crops = model(images)
        logits1, logits2, logits3 = logits

        classification_loss1 = criterion(logits1, labels)
        classification_loss2 = criterion(logits2, labels)
        classification_loss3 = criterion(logits3, labels)

        rank_loss1 = rank_loss(
            logits_coarse=logits1, 
            logits_fine=logits2, 
            labels=labels
        )
        rank_loss2 = rank_loss(
            logits_coarse=logits3, 
            logits_fine=logits1, 
            labels=labels,
        )

        loss = classification_loss1 + classification_loss2 + classification_loss3 + rank_loss1 + rank_loss2
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        total += labels.size(dim=0)

        prediction1 = logits1.argmax(dim=1)
        prediction2 = logits2.argmax(dim=1)
        _, prediction3 = logits3.max(dim=1)
        correct1 = prediction1.eq(labels).sum().item()
        correct2 = prediction2.eq(labels).sum().item()
        correct3 = prediction3.eq(labels).sum().item()

        if (index + 1) % 100 == 0:
            print(f"Step: {(index + 1) /len(train_loader)}, Loss: {loss.item():.4f}")

    print(f"Loss: {running_loss/len(train_loader):.4f} "
          f"Acc1: {100.*correct1/total:.2f}% "
          f"Acc2: {100.*correct2/total:.2f}% "
          f"Acc3: {100.*correct3/total:.2f}%")